# GSE270828 — Stage 7 Rescue (No Google Drive)

This is a **format-aware repair** after the original full `GSE270828_Merged_CountMatrix.rds.gz` download succeeded but `readRDS()` returned **unknown input format**.

1. Reuse existing file from the **same Colab runtime**, if present. 2. Otherwise download it again **directly from NCBI GEO**. 3. Identify the real content format from its **uncompressed header**, instead of trusting the filename. 4. Attempt the appropriate R parser only if supported. 5. Export cell metadata and rigorous QC only — no differential expression or claims of functional validation.

All files stay within `/content`. **No Drive mounting, authentication or personal account access.**

If the RDS is a genuine large R object, extraction may still require a high-memory Colab runtime. Invalid content is reported rather than misinterpreted as biology.

In [ ]:
# Dependencies (Python only; R added automatically if needed)
import gzip, json, os, shutil, subprocess, sys, time, zipfile, traceback
from pathlib import Path
import pandas as pd
import requests
print('Python:',sys.version.split()[0])

In [ ]:
"""GSE270828 Colab preflight, direct NCBI delivery, metadata-only QC.

No Google Drive account, mount, token, or API is involved. All analyses
are descriptive. No differential expression or causal inference is run.
"""
from __future__ import annotations

import csv
import gzip
import hashlib
import json
import os
import re
import shutil
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

GEO_BASE = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828"
GUIDE_URL = GEO_BASE + "/suppl/GSE270828_feature_README.csv"
MATRIX_URL = GEO_BASE + "/matrix/GSE270828_series_matrix.txt.gz"
RDS_URL = GEO_BASE + "/suppl/GSE270828_Merged_CountMatrix.rds.gz"
EXPECTED_RDS_SIZE_BYTES = 2312020968  # observed original file inventory, recheck via HEAD
CONTROL_RE = re.compile(r"(?:^ntc(?:\d|_|\b)|non[-_ ]?target|negative[-_ ]?control|^cont[_-]|^ctrl\b|scrambl)", re.I)


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def checksum_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def download_small(url, dest, timeout=45):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, timeout=(20, timeout), stream=True, headers={'User-Agent':'HAR-Stage7-Research/1.0'}) as r:
        r.raise_for_status()
        with dest.open('wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk:
                    f.write(chunk)
    if dest.stat().st_size < 100:
        raise RuntimeError(f"Unexpected tiny file: {dest}")
    return dest


def audit_guide(guide_csv, out_dir):
    out = Path(out_dir)
    out.mkdir(exist_ok=True, parents=True)
    df = pd.read_csv(guide_csv, dtype='string')
    required = {'id', 'name', 'sequence', 'feature_type', 'target_gene_name'}
    if not required.issubset(df.columns):
        raise ValueError('Unexpected guide file columns: ' + ', '.join(map(str, df.columns)))
    df['control_guide'] = df['id'].str.match(r'^(?:NTC\d+|NTC_10x|cont[_-])', case=False, na=False)
    df['HAR_screened'] = ~df['control_guide']
    har = df.loc[df['HAR_screened']].groupby('name', dropna=False).agg(
        n_guides=('id','size'), distinct_sequences=('sequence','nunique'),
        design_target_names=('target_gene_name', lambda x:'; '.join(sorted(x.dropna().unique()))),
    ).reset_index().rename(columns={'name':'HAR'})
    har.to_csv(out/'design_HAR_guide_counts.tsv', sep='\t', index=False)
    df.loc[df['control_guide']].to_csv(out/'guide_controls.tsv', sep='\t', index=False)
    (out/'screened_HAR_labels.txt').write_text('\n'.join(sorted(har['HAR'].astype(str)))+'\n',encoding='utf-8')
    summary = {
        'source_url': GUIDE_URL,
        'guide_rows_total': len(df),
        'designed_HARs': len(har),
        'HAR_guides': int(df['HAR_screened'].sum()),
        'control_guides_total': int(df['control_guide'].sum()),
        'non_targeting_guide_count': int(df['id'].str.match(r'^NTC(?:\d+|_10x)', case=False, na=False).sum()),
        'design_annotations_are_not_validated_gene_targets': True,
        'sha256_original_guide': checksum_sha256(guide_csv),
    }
    write_json(out/'guide_design_qc.json', summary)
    return summary, sorted(har['HAR'].astype(str).tolist())


def audit_geo_series_matrix(matrix_gz, out_dir):
    out = Path(out_dir)
    out.mkdir(parents=True,exist_ok=True)
    fields = {}
    with gzip.open(matrix_gz, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!Sample_'):
                row = next(csv.reader([line.rstrip('\r\n')], delimiter='\t'))
                fields[row[0]] = row[1:]
    if not fields:
        raise ValueError('No !Sample_* metadata in GEO series matrix')
    sizes = Counter(len(x) for x in fields.values())
    if len(sizes)>1:
        raise ValueError(f'GEO sample metadata row sizes differ: {sizes}')
    sample_count = next(iter(sizes))
    if sample_count > 0:
        dat = pd.DataFrame({col:values for col,values in fields.items()})
        dat.insert(0,'GEO_metadata_index',range(1,len(dat)+1))
        dat.to_csv(out/'GEO_sample_metadata.tsv',sep='\t',index=False)
    summary = {'GEO_sample_metadata_entries':int(sample_count),'GEO_sample_fields':list(fields),
               'sample_rows_are_not_independent_donors':True,
               'sha256_original_matrix':checksum_sha256(matrix_gz)}
    write_json(out/'geo_sample_metadata_qc.json', summary)
    return summary


def get_cgroup_memory_bytes():
    for p in ('/sys/fs/cgroup/memory.max','/sys/fs/cgroup/memory/memory.limit_in_bytes'):
        try:
            s=Path(p).read_text().strip()
            if s!='max':
                n=int(s)
                if 0<n<2**50:return n
        except (OSError,ValueError):pass
    try:
        import psutil
        return int(psutil.virtual_memory().total)
    except Exception:
        return None


def head_remote_size(url):
    try:
        with requests.head(url, allow_redirects=True, timeout=(15,30)) as r:
            r.raise_for_status()
            return int(r.headers['Content-Length']) if r.headers.get('Content-Length','').isdigit() else None
    except requests.RequestException:
        return None


def download_large_resumable(url, dest, min_free_extra=3*1024**3):
    """Stream file to /content, resume only if server honors byte ranges.

    Full-size check is necessary; this function does NOT parse or authenticate the RDS.
    """
    dest=Path(dest);dest.parent.mkdir(parents=True, exist_ok=True)
    expected=head_remote_size(url) or EXPECTED_RDS_SIZE_BYTES
    free=shutil.disk_usage(dest.parent).free
    existing=dest.stat().st_size if dest.exists() else 0
    if existing==expected:
        return {'status':'already_downloaded','bytes':existing,'remote_expected':expected,'source':url}
    partial=dest.with_name(dest.name+'.part')
    size=partial.stat().st_size if partial.exists() else 0
    needed=max(0,expected-size)
    if free<needed+min_free_extra:
        raise RuntimeError(f'Insufficient free local storage: need about {(needed+min_free_extra)/1e9:.1f} GB, have {free/1e9:.1f} GB.')
    headers={'User-Agent':'HAR-Stage7-Research/1.0'}
    if size: headers['Range']=f'bytes={size}-'
    with requests.get(url,headers=headers,stream=True,timeout=(30,120)) as r:
        r.raise_for_status()
        if size and r.status_code==206:
            content_range=r.headers.get('Content-Range','')
            if not content_range.startswith(f'bytes {size}-'):
                raise RuntimeError(f'Resumed response is not aligned with partial file: {content_range}')
            mode='ab'
        elif size and r.status_code==200:
            size=0;mode='wb'
        else:
            mode='wb'
        with partial.open(mode) as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk:f.write(chunk)
    actual=partial.stat().st_size
    if expected and actual!=expected:
        raise RuntimeError(f'Incomplete original GEO RDS: got {actual:,} bytes, expected {expected:,}. Re-run cell to resume.')
    partial.replace(dest)
    return {'status':'downloaded','bytes':actual,'remote_expected':expected,'source':url}


def audit_cell_metadata(meta_tsv, out_dir, screened_hars, source='GSE270828 original RDS'):
    """Metadata-only, donor-aware cautions. No DE analysis."""
    out=Path(out_dir); out.mkdir(parents=True,exist_ok=True)
    obs=pd.read_csv(meta_tsv,sep='\t',low_memory=False)
    for c in ('har','guide','replicate','rep','lane','batch','perturbation_name'):
        if c in obs: obs[c]=obs[c].astype('string').fillna('<missing>').str.strip()
    schema=pd.DataFrame([{
        'field':k,'dtype':str(obs[k].dtype),'missing_count':int(obs[k].isna().sum()),
        'distinct_nonmissing_values':int(obs[k].nunique(dropna=True))} for k in obs.columns])
    schema.to_csv(out/'metadata_schema.tsv',sep='\t',index=False)
    result={'source':source,'n_cells_metadata_rows':len(obs),'metadata_columns':list(obs.columns),
            'has_har_label':'har' in obs,'has_guide_label':'guide' in obs,
            'has_biological_donor_identifiers':False,'donor_identity_not_assumed_from_replicate_or_lane':True,
            'no_statistical_effects_computed':True}
    if 'har' not in obs:
        result['qc_status']='missing HAR assignment column; no HAR coverage calculated'
        write_json(out/'single_cell_metadata_summary.json',result)
        return result
    hc=obs.groupby('har',dropna=False).size().rename('n_cells').reset_index().sort_values('n_cells',ascending=False)
    hc['possible_control_label_not_confirmed']=hc['har'].map(lambda x:bool(CONTROL_RE.search(str(x))))
    hc.to_csv(out/'har_cell_counts.tsv',sep='\t',index=False)
    controls=hc.loc[hc['possible_control_label_not_confirmed']]
    controls.to_csv(out/'possible_control_har_labels.tsv',sep='\t',index=False)
    labels=set(map(str,screened_hars))
    cover=pd.DataFrame({'HAR':sorted(labels)}).merge(hc[['har','n_cells']],how='left',left_on='HAR',right_on='har').drop(columns=['har'])
    cover['n_cells']=cover['n_cells'].fillna(0).astype('int64')
    cover['observed_in_metadata']=cover['n_cells']>0
    replicate_col=next((c for c in ['replicate','rep'] if c in obs),None)
    for col,filename in [('guide','har_by_guide.tsv'),(replicate_col,'har_by_replicate_label.tsv'),('lane','har_by_lane.tsv'),('batch','har_by_batch.tsv')]:
        if col and col in obs.columns:
            tab=obs.groupby(['har',col],dropna=False).size().rename('n_cells').reset_index()
            tab.to_csv(out/filename,sep='\t',index=False)
            cnt=obs.groupby('har')[col].nunique().rename('n_distinct_'+col+'_labels')
            cover=cover.join(cnt,on='HAR')
            cover['n_distinct_'+col+'_labels']=cover['n_distinct_'+col+'_labels'].fillna(0).astype('int64')
    cover.to_csv(out/'screened_HAR_coverage.tsv',sep='\t',index=False)
    result.update({'qc_status':'metadata successfully audited',
                   'n_HARs_in_original_design':len(labels),
                   'n_HARs_observed_in_metadata':int(cover['observed_in_metadata'].sum()),
                   'n_unique_har_labels_in_metadata':int(hc.shape[0]),
                   'n_possible_control_har_labels_heuristic_only':int(len(controls)),
                   'replicate_label_col':replicate_col,
                   'n_unique_replicate_labels':int(obs[replicate_col].nunique()) if replicate_col else None,
                   'n_unique_lane_labels':int(obs['lane'].nunique()) if 'lane' in obs else None})
    write_json(out/'single_cell_metadata_summary.json',result)
    return result


In [ ]:
# Configuration. Set False to inspect small metadata only, if large re-download is not feasible.
DOWNLOAD_IF_MISSING = True  # @param {type:'boolean'}
ATTEMPT_R_EXTRACTION = True  # @param {type:'boolean'}
MIN_RAM_GB = 9.5  # @param {type:'number'}
BASE=Path('/content') if Path('/content').is_dir() else Path.cwd()
DATA=BASE/'GSE270828_public_source'
OUT=BASE/'HAR_Stage7_Rescue_Output'
DATA.mkdir(parents=True,exist_ok=True);OUT.mkdir(parents=True,exist_ok=True)
SOURCE=DATA/'GSE270828_Merged_CountMatrix.rds.gz'
META=DATA/'GSE270828_original_cell_metadata_rescue.tsv'
STATUS={'stage':'Stage 7 Rescue', 'source':'NCBI GEO GSE270828', 'google_drive_used':False,
        'differential_expression_performed':False, 'original_source_size_expected':EXPECTED_RDS_SIZE_BYTES}
print('Prior data file exists?', SOURCE.exists())
if SOURCE.exists():print('Bytes on disk:',SOURCE.stat().st_size)
print('RAM limit GB:', round(get_cgroup_memory_bytes()/1e9,2) if get_cgroup_memory_bytes() else 'unknown')
print('Free disk GB:',round(shutil.disk_usage(DATA).free/1e9,2))

In [ ]:
# Source-file diagnostic & download only if needed
try:
    if SOURCE.exists() and SOURCE.stat().st_size==EXPECTED_RDS_SIZE_BYTES:
        info={'status':'reused_existing','bytes':SOURCE.stat().st_size}
    elif DOWNLOAD_IF_MISSING:
        info=download_large_resumable(RDS_URL,SOURCE,min_free_extra=2*1024**3)
    else:
        info={'status':'missing_or_wrong_size_no_redownload', 'exists':SOURCE.exists(),
              'bytes':SOURCE.stat().st_size if SOURCE.exists() else None}
    STATUS['original_file']=info
except Exception as exc:
    STATUS['original_file']={'status':'error','error':str(exc)}
write_json(OUT/'STATUS.json',STATUS)
print(json.dumps(STATUS['original_file'],indent=2))

## Verify compressed-file identity

The crucial diagnostic step. When gzip decompression succeeds, `RDX3`/`RDA3` at the beginning indicates an R workspace (`load()`), whereas `X\n` or `A\n` indicates an R serialized object (`readRDS()`). Unknown headers, nested compression and corrupt data are **not** silently coerced.

Valid first bytes do **not** prove full-file integrity; optional CRC check appears below.

In [ ]:
def sniff_source(path):
    path=Path(path)
    if not path.is_file():return {'status':'missing','format':'unknown'}
    result={'status':'checked','compressed_bytes':path.stat().st_size}
    with path.open('rb') as inp: raw=inp.read(12)
    result['raw_first12_hex']=raw.hex()
    result['outer_gzip_magic']=raw.startswith(b'\x1f\x8b')
    try:
        with gzip.open(path,'rb') as gz: head=gz.read(64)
    except Exception as exc:
        result.update({'format':'invalid_or_not_gzip','gzip_header_error':str(exc)})
        return result
    result['uncompressed_first64_hex']=head.hex()
    result['uncompressed_first64_ascii']=repr(head)
    if head.startswith((b'RDX2\n',b'RDX3\n',b'RDA2\n',b'RDA3\n',b'RDB2\n',b'RDB3\n')):
        fmt='R_workspace'
    elif head.startswith((b'X\n',b'A\n',b'B\n')):
        fmt='RDS'
    elif head.startswith(b'\x1f\x8b'):
        fmt='nested_gzip'
    else:
        fmt='unknown'
    result['format']=fmt
    return result

SNIFF=sniff_source(SOURCE)
STATUS['signature_inspection']=SNIFF
write_json(OUT/'RDS_format_signature.json',SNIFF)
write_json(OUT/'STATUS.json',STATUS)
print(json.dumps(SNIFF,indent=2))

In [ ]:
# Optional full gzip CRC validation; decompresses stream without loading it into RAM
# ENABLE_GZIP_INTEGRITY_SCAN = False by default to save time (full read is expensive).
ENABLE_GZIP_INTEGRITY_SCAN = False  # @param {type:'boolean'}
if ENABLE_GZIP_INTEGRITY_SCAN and SOURCE.is_file():
    try:
        decoded=0
        with gzip.open(SOURCE,'rb') as f:
            while True:
                b=f.read(8*1024*1024)
                if not b:break
                decoded+=len(b)
        STATUS['full_gzip_integrity']={'status':'pass','decompressed_bytes':decoded}
    except Exception as exc:
        STATUS['full_gzip_integrity']={'status':'failed','message':str(exc)}
else:
    STATUS['full_gzip_integrity']={'status':'not_run'}
write_json(OUT/'STATUS.json',STATUS)
print(STATUS['full_gzip_integrity'])

## Extract only cell-level metadata with format-aware R parser

The script uses `load()` for R workspace files, `readRDS()` for serialized RDS, and refuses to parse unknown formats. Even for recognized formats, memory or package/class incompatibilities can prevent deserialization. Results remain unverified until real output is produced.

In [ ]:
R_SOURCE = "args <- commandArgs(trailingOnly=TRUE)\nsuppressPackageStartupMessages(library(methods))\nif (length(args) < 4) stop('Expected: source output.tsv log.txt format_label')\nf <- args[[1]]; output <- args[[2]]; logpath <- args[[3]]; fmt <- args[[4]]\nlog_msg <- function(...) {cat(..., '\\n', file=logpath, append=TRUE)}\nlog_msg('Begin original object extraction. Format=',fmt)\ncon <- gzfile(f, open='rb')\n# The R process is short-lived; explicitly close the connection after deserialization.\nif (fmt == 'R_workspace') {\n  env <- new.env(parent=baseenv())\n  objs <- load(con, envir=env)\n  log_msg('load() recovered object names:', paste(objs, collapse=','))\n  if (length(objs) < 1) stop('No objects found in R workspace')\n  vals <- mget(objs,envir=env,inherits=FALSE)\n  candidate <- vapply(vals, function(obj) {\n      (isS4(obj) && 'meta.data' %in% slotNames(obj)) ||\n      (is.list(obj) && 'meta.data' %in% names(obj)) || is.data.frame(obj)\n  }, logical(1))\n  if (sum(candidate)!=1) stop(paste('Need exactly one usable metadata-bearing object, found',sum(candidate), 'among:',paste(objs,collapse=',')))\n  obj <- vals[[which(candidate)[1]]]\n} else if (fmt == 'RDS') {\n  obj <- readRDS(con)\n} else {\n  stop(paste('Unsupported format:',fmt))\n}\nclose(con)\nlog_msg('Object class:',paste(class(obj),collapse=','))\nif (isS4(obj) && 'meta.data' %in% slotNames(obj)) {\n  md <- methods::slot(obj,'meta.data')\n} else if (is.list(obj) && 'meta.data' %in% names(obj)) {\n  md <- obj[['meta.data']]\n} else if (is.data.frame(obj)) {\n  md <- obj\n} else {\n  stop('No direct meta.data slot, list element, or data.frame. Cannot export cell-level metadata without additional class-aware analysis.')\n}\nif (!is.data.frame(md)) stop('Metadata is not a data.frame')\nlog_msg('Metadata dimensions',nrow(md),ncol(md))\nmd <- data.frame(cell_barcode = rownames(md), md, check.names=FALSE, stringsAsFactors=FALSE)\nutils::write.table(md, file=output, sep='\\t', quote=TRUE, row.names=FALSE, na='')\nlog_msg('SUCCESS: exported',nrow(md),'cell metadata rows')\n"
R_SCRIPT_FILE=BASE/'stage7_rescue_extract_metadata.R'
R_SCRIPT_FILE.write_text(R_SOURCE,encoding='utf-8')
try:
    detected=SNIFF.get('format')
    if not ATTEMPT_R_EXTRACTION:
        raise RuntimeError('R extraction disabled by user')
    if detected not in {'RDS','R_workspace'}:
        raise RuntimeError('Unsupported or invalid detected serialization format: '+str(detected))
    mem=get_cgroup_memory_bytes()
    if mem and mem/1e9<MIN_RAM_GB:
        raise RuntimeError('Memory limit '+str(round(mem/1e9,2))+' GB is below configured safe threshold '+str(MIN_RAM_GB)+' GB')
    if not shutil.which('Rscript'):
        print('Installing base R for object extraction, without Seurat package...')
        inst=subprocess.run(['bash','-lc','apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-base-core'],
                            capture_output=True,text=True,timeout=1200)
        (OUT/'R_install_log.txt').write_text((inst.stdout+'\n'+inst.stderr)[-12000:])
        if inst.returncode!=0:raise RuntimeError('R installation failed. See R_install_log.txt')
    def ram_limit():
        import resource
        n=get_cgroup_memory_bytes()
        if n and sys.platform.startswith('linux'):
            cap=int(n*0.92);resource.setrlimit(resource.RLIMIT_AS,(cap,cap))
    proc=subprocess.run(['Rscript',str(R_SCRIPT_FILE),str(SOURCE),str(META),str(OUT/'R_extractor_progress.txt'),detected],
                       capture_output=True,text=True,timeout=2700,preexec_fn=ram_limit)
    (OUT/'R_extractor_stdout_stderr.txt').write_text((proc.stdout+'\n--- STDERR ---\n'+proc.stderr)[-16000:])
    if proc.returncode or not META.exists() or META.stat().st_size<100:
        raise RuntimeError('R parser return code: '+str(proc.returncode)+'; metadata TSV unavailable. Check R_extractor_stdout_stderr.txt')
    STATUS['metadata_extraction']={'status':'success','bytes':META.stat().st_size,'detected_format':detected}
    print('Metadata TSV exported successfully:',META.stat().st_size,'bytes')
except Exception as exc:
    STATUS['metadata_extraction']={'status':'error_or_skipped','message':str(exc)}
    print('Metadata extraction not completed:',exc)
write_json(OUT/'STATUS.json',STATUS)

## Real metadata QC: only if cell metadata extracted

In [ ]:
# Small source guide is re-downloaded only if absent; no Google Drive.
try:
    guide=DATA/'GSE270828_feature_README.csv'
    if not guide.exists():download_small(GUIDE_URL,guide)
    design_summary,screened=audit_guide(guide,OUT)
    STATUS['guide_design']=design_summary
    if META.exists() and META.stat().st_size>100:
        STATUS['cell_level_qc']=audit_cell_metadata(META,OUT,screened,source='Original GEO R file; Stage 7 Rescue')
        print('Cell-level QC complete:',STATUS['cell_level_qc'].get('n_cells_metadata_rows'))
    else:
        STATUS['cell_level_qc']={'status':'not_run','reason':'No exported cell-level metadata'}
except Exception as exc:
    STATUS['cell_level_qc']={'status':'error','message':str(exc)}
    print('QC error:',exc)
write_json(OUT/'STATUS.json',STATUS)

In [ ]:
# Download small diagnostic results ZIP, never the multi-GB original.
resultzip=BASE/'HAR_Stage7_Rescue_Results.zip'
with zipfile.ZipFile(resultzip,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for p in OUT.rglob('*'):
        if p.is_file():archive.write(p,arcname=p.relative_to(OUT).as_posix())
print('Results ZIP:',resultzip,'bytes:',resultzip.stat().st_size)
try:
    from google.colab import files
    files.download(str(resultzip))
except ImportError:
    print('Not on Colab; file saved locally:',resultzip)
except Exception as exc:
    print('Auto-download not started:',exc,'Use Colab Files panel to download the ZIP.')